# Setup

In [1]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)
documents = [file.parse() for file in reader.read()]
len(documents)

72

# Q1. Generating questions

In [2]:
from pydantic import BaseModel


class Questions(BaseModel):
    questions: list[str]

In [3]:
data_gen_instructions = """
You emulate a student who is taking our LLM course.
You are given one lesson page from the course.
Formulate 5 questions this student might ask that are answered by this page.

Rules:
- The page should contain the answer to each question.
- Make the questions complete and not too short.
- Use as few words as possible from the page; don't copy its phrasing.
- The questions should resemble how people actually ask things online:
  not too formal, not too short, not too long.
- Ask about the content of the lesson, not about its formatting or filename.
""".strip()

In [4]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [5]:
import json

from evaluation_utils import llm_structured

pages = documents[:3]
input_tokens = []

for doc in pages:
    user_prompt = json.dumps({"filename": doc["filename"], "content": doc["content"]})
    result, usage = llm_structured(openai_client, data_gen_instructions, user_prompt, Questions)
    print(doc["filename"], usage.input_tokens)
    input_tokens.append(usage.input_tokens)

sum(input_tokens) / len(input_tokens)

01-agentic-rag/lessons/01-intro.md 1021


01-agentic-rag/lessons/02-environment.md 1287


01-agentic-rag/lessons/03-rag.md 1754


1354.0

# The full ground truth

In [6]:
import pandas as pd

df_ground_truth = pd.read_csv("data/ground-truth.csv")
ground_truth = df_ground_truth.to_dict(orient="records")
len(ground_truth)

360

# Searching the chunks

In [7]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)
len(chunks)

295

In [8]:
import sys

sys.path.insert(0, "../module-02")

from embedder import Embedder

embedder = Embedder(path="../module-02/models/Xenova/all-MiniLM-L6-v2")
X = embedder.encode_batch([c["content"] for c in chunks])

In [9]:
from minsearch import Index, VectorSearch

text_index = Index(text_fields=["content"], keyword_fields=["filename"])
text_index.fit(chunks)

vector_index = VectorSearch(keyword_fields=["filename"])
vector_index.fit(X, chunks)

In [10]:
def text_search(query, num_results=5):
    return text_index.search(query, num_results=num_results)


def vector_search(query, num_results=5):
    v = embedder.encode(query)
    return vector_index.search(v, num_results=num_results)

# Q2. First result with text search

In [11]:
q = ground_truth[0]["question"]
text_search(q)[0]["filename"]

'01-agentic-rag/lessons/03-rag.md'

# Q3. First result with vector search

In [12]:
vector_search(q)[0]["filename"]

'01-agentic-rag/lessons/01-intro.md'

# Evaluation metrics

In [13]:
from tqdm.auto import tqdm


def compute_relevance(q, search_function):
    filename = q["filename"]
    results = search_function(query=q["question"])
    return [int(d["filename"] == filename) for d in results]


def compute_relevance_total(ground_truth, search_function):
    return [compute_relevance(q, search_function) for q in tqdm(ground_truth)]


def hit_rate(relevance):
    return sum(1 for line in relevance if 1 in line) / len(relevance)


def mrr(relevance):
    total_score = 0.0
    for line in relevance:
        for rank in range(len(line)):
            if line[rank] == 1:
                total_score += 1 / (rank + 1)
                break
    return total_score / len(relevance)


def evaluate(ground_truth, search_function):
    relevance_total = compute_relevance_total(ground_truth, search_function)
    return {"hit_rate": hit_rate(relevance_total), "mrr": mrr(relevance_total)}

# Q4. Evaluating text search

In [14]:
evaluate(ground_truth, text_search)

  0%|          | 0/360 [00:00<?, ?it/s]

{'hit_rate': 0.7583333333333333, 'mrr': 0.5942592592592594}

# Q5. Evaluating vector search

In [15]:
evaluate(ground_truth, vector_search)

  0%|          | 0/360 [00:00<?, ?it/s]

{'hit_rate': 0.725, 'mrr': 0.5486111111111112}

# Q6. Tuning hybrid search

In [16]:
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]


def hybrid_search(query, k=60, num_results=5):
    text_results = text_search(query, num_results=10)
    vector_results = vector_search(query, num_results=10)
    return rrf([text_results, vector_results], k=k, num_results=num_results)

In [17]:
for k in [1, 50, 100, 200]:
    result = evaluate(ground_truth, lambda query, k=k: hybrid_search(query, k=k))
    print(f"k={k}: {result}")

  0%|          | 0/360 [00:00<?, ?it/s]

k=1: {'hit_rate': 0.8388888888888889, 'mrr': 0.6481944444444449}


  0%|          | 0/360 [00:00<?, ?it/s]

k=50: {'hit_rate': 0.8361111111111111, 'mrr': 0.637916666666667}


  0%|          | 0/360 [00:00<?, ?it/s]

k=100: {'hit_rate': 0.8361111111111111, 'mrr': 0.637916666666667}


  0%|          | 0/360 [00:00<?, ?it/s]

k=200: {'hit_rate': 0.8361111111111111, 'mrr': 0.637916666666667}
